# Find Signal/Background Cross-Sections


In [7]:
import pythia8
import datetime

date_time = datetime.datetime.now()
time_seed = date_time.time().strftime('%H%M%S')

LUMI_FB      = 110.0
DRY_RUN_EVTS = 1000    

In [8]:
def get_sigma(setup_fn, n_events=DRY_RUN_EVTS):

    pythia = pythia8.Pythia()
    setup_fn(pythia)
    pythia.readString("Print:quiet = on")   
    pythia.init()

    for _ in range(n_events):
        pythia.next()

    sigma_fb = pythia.infoPython().sigmaGen() * 1e12
    sigma_err_fb = pythia.infoPython().sigmaErr() * 1e12
    pythia.stat()
    return sigma_fb, sigma_err_fb

In [9]:
# ── Copy your setup functions here ────────────────────────────────────────

def setup_signal(pythia):
    pythia.readString("Beams:idA = 2212")
    pythia.readString("Beams:idB = 2212")
    pythia.readString("Beams:eCM = 13600.")
    pythia.readString("HiggsSM:all = on")
    pythia.readString("25:onMode = off")
    pythia.readString("25:onIfMatch = 23 23")
    pythia.readString("23:onMode = off")
    pythia.readString("23:onIfMatch = 13 -13")
    pythia.readString("Random:setSeed = on")
    pythia.readString(f"Random:seed = {time_seed}")


def setup_background(pythia):
    pythia.readString("Beams:idA = 2212")
    pythia.readString("Beams:idB = 2212")
    pythia.readString("Beams:eCM = 13600.")
    pythia.readString("WeakDoubleBoson:ffbar2gmZgmZ = on")
    pythia.readString("WeakZ0:gmZmode = 0")
    pythia.readString("23:onMode = off")
    pythia.readString("23:onIfMatch = 13 -13")
    pythia.readString("Random:setSeed = on")
    pythia.readString(f"Random:seed = {time_seed}")

In [10]:
sigma_sig, err_sig = get_sigma(setup_signal)
sigma_bkg, err_bkg = get_sigma(setup_background)



 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.312      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 23 May 2024  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T    H   H  III  A   A    Now is 16 May 2026 at 11:40:40    |  | 
 |  |                                                                              |  | 
 |  |   Program docu

In [11]:
ratio = sigma_bkg / sigma_sig

print("\n" + "=" * 55)
print(f"  sigma (signal)     = {sigma_sig:.4e} ± {err_sig:.2e} fb")
print(f"  sigma (background) = {sigma_bkg:.4e} ± {err_bkg:.2e} fb")
print(f"  sigma_bkg / sigma_sig = {ratio:.1f}")
print("=" * 55)

yield_sig = sigma_sig * LUMI_FB
yield_bkg = sigma_bkg * LUMI_FB
print(f"\n  Expected signal events  {yield_sig:.2f}")
print(f"  Expected background events {yield_bkg:.2f}")


  sigma (signal)     = 1.0255e+00 ± 2.28e-02 fb
  sigma (background) = 1.7995e+01 ± 2.45e-01 fb
  sigma_bkg / sigma_sig = 17.5

  Expected signal events  112.81
  Expected background events 1979.46


In [12]:
TOTAL_SIM = 100000  

frac_sig = (1 / sigma_sig) / (1 / sigma_sig + 1 / sigma_bkg)
frac_bkg = 1 - frac_sig

# print(frac_sig)

N_sig_rec = int(TOTAL_SIM * frac_sig)
N_bkg_rec = int(TOTAL_SIM * frac_bkg)

print("\n" + "=" * 55)
print(f"  Recommended N_signal     = {N_sig_rec:,}")
print(f"  Recommended N_background = {N_bkg_rec:,}")
print(f"  Ratio  N_sig / N_bkg     = {N_sig_rec / max(N_bkg_rec,1):.1f}")
print(f"  signal-to-bkg ratio for production     = {frac_sig:.5f}")
print("=" * 55)
print()
print("Paste these into Cell 4 of analysis_2b.ipynb:")
print(f"  run('signal',     n_events={N_sig_rec}, out_file='signal.root')")
print(f"  run('background', n_events={N_bkg_rec}, out_file='background.root')")


  Recommended N_signal     = 94,608
  Recommended N_background = 5,391
  Ratio  N_sig / N_bkg     = 17.5
  signal-to-bkg ratio for production     = 0.94608

Paste these into Cell 4 of analysis_2b.ipynb:
  run('signal',     n_events=94608, out_file='signal.root')
  run('background', n_events=5391, out_file='background.root')
